# App 1 — Market Basket Analysis với AprioriFIM

Notebook trình diễn và tái hiện thực nghiệm Market Basket Analysis trên bộ dữ liệu **Groceries**. Notebook chỉ điều phối việc nạp dữ liệu và gọi lại các hàm đã có trong package `AprioriFIM` cùng ứng dụng `applications/market_basket.jl`; **không cài đặt lại thuật toán Apriori**.

**Cách chạy:** mở notebook bằng Google Colab (không cần đăng nhập GitHub), chọn *Runtime → Run all*. Dùng Python runtime mặc định. Cell thiết lập sẽ tự cài Julia qua `juliacall`, clone mã nguồn công khai và kích hoạt project. Lần chạy đầu cần Internet và có thể mất vài phút để tải Julia.

## 1. Giới thiệu bài toán

Market Basket Analysis tìm các nhóm sản phẩm thường xuất hiện cùng nhau trong một giao dịch. Quy trình demo:

**Dữ liệu giao dịch → Frequent Itemset Mining → Sinh luật kết hợp → Lọc/xếp hạng → Diễn giải kinh doanh.**

Một luật có dạng `X ⇒ Y`, trong đó hai vế không giao nhau:

- **Support:** tỷ lệ giao dịch chứa đồng thời `X` và `Y`.
- **Confidence:** tỷ lệ giao dịch chứa `Y` trong số các giao dịch đã chứa `X`.
- **Lift:** so sánh Confidence với mức phổ biến nền của `Y`. `Lift > 1` cho thấy hai vế đồng xuất hiện nhiều hơn mức kỳ vọng nếu độc lập.

Luật kết hợp mô tả mối liên hệ thống kê, không chứng minh quan hệ nhân quả.

## 2. Thiết lập môi trường Julia và nạp package

Notebook dùng Python runtime mặc định để tự tải Julia qua `juliacall`. Julia kích hoạt trực tiếp `Project.toml` của repository và nạp `AprioriFIM`.

Repository được ghim tại commit `f965de2597b6d1564fd3a0440b14e78c0e1cdf0e` để người chạy sau có thể tái hiện cùng phiên bản mã nguồn. Không cần cài thư viện khai phá luật bên ngoài.

In [ ]:
import os, sys, subprocess, pathlib, hashlib
from IPython.display import display, Markdown
import pandas as pd

REPO_URL = "https://github.com/TaVanDuc2005/FIM_Group14.git"
REPO_COMMIT = "f965de2597b6d1564fd3a0440b14e78c0e1cdf0e"
REPO_DIR = pathlib.Path("/content/FIM_Group14")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_COMMIT], check=True)

required = [
    REPO_DIR / "Project.toml",
    REPO_DIR / "src" / "AprioriFIM.jl",
    REPO_DIR / "applications" / "market_basket.jl",
    REPO_DIR / "data" / "market" / "groceries.csv",
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Repository thiếu tệp cần thiết: " + ", ".join(missing))

print("Repository:", REPO_DIR)
print("Commit:", subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip())

# juliacall tự tải/cài Julia tương thích nếu runtime chưa có Julia.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "juliacall"], check=True)
from juliacall import Main as jl

repo_julia = str(REPO_DIR).replace("\\", "/")
jl.seval(f'''
using Pkg
Pkg.activate(raw"{repo_julia}")
using AprioriFIM
include(raw"{repo_julia}/applications/market_basket.jl")
println("Julia version: ", VERSION)
println("Project: ", Base.active_project())
println("AprioriFIM loaded successfully.")
''')

## 3. Đọc dữ liệu Groceries và kiểm tra thống kê

Dữ liệu nằm sẵn trong repository: mỗi dòng là một giỏ hàng, các mặt hàng phân tách bằng dấu phẩy. Notebook sử dụng `load_groceries` của App 1, trong đó gọi `read_labeled_transactions` và `encode_transactions` của package.

Bản dữ liệu được kiểm tra theo 9.835 giao dịch, 169 mặt hàng và SHA-256 đã ghi trong `data/market/README.md`.

In [ ]:
data_path = REPO_DIR / "data" / "market" / "groceries.csv"
expected_sha256 = "f8a7074231acf5f3de6bdb7ec63fed03c28745d76bd49aa8ddc0c5d1cfa3da91"

# Chuẩn hóa CRLF về LF chỉ để kiểm tra checksum; không sửa file gốc.
normalized = b"\n".join(line.rstrip(b"\r") for line in data_path.read_bytes().splitlines()) + b"\n"
actual_sha256 = hashlib.sha256(normalized).hexdigest()
if actual_sha256 != expected_sha256:
    raise ValueError(f"Checksum dữ liệu không khớp: {actual_sha256}")

data_path_julia = str(data_path).replace("\\", "/")
jl.seval(f'transactions, labels = load_groceries(raw"{data_path_julia}")')

n_transactions = int(jl.seval("length(transactions)"))
n_items = int(jl.seval("length(labels)"))
basket_sizes = list(jl.seval("[length(t) for t in transactions]"))
total_item_occurrences = sum(basket_sizes)

stats = pd.DataFrame([
    ("Số giao dịch", n_transactions),
    ("Số mặt hàng phân biệt", n_items),
    ("Tổng lượt xuất hiện mặt hàng", total_item_occurrences),
    ("Kích thước giỏ trung bình", round(sum(basket_sizes) / n_transactions, 3)),
    ("Kích thước giỏ lớn nhất", max(basket_sizes)),
    ("Giỏ chỉ có 1 mặt hàng", sum(x == 1 for x in basket_sizes)),
    ("SHA-256", actual_sha256),
], columns=["Chỉ số", "Giá trị"])
display(stats)

## 4. Khảo sát tham số — 35 tổ hợp `minsup × minconf`

Gọi trực tiếp `sweep_thresholds(transactions)` trong `applications/market_basket.jl`. Hàm thực hiện 7 mức `minsup` × 5 mức `minconf`, tổng cộng **35 tổ hợp**. Với mỗi mức `minsup`, Apriori chỉ chạy một lần; các mức `minconf` được khảo sát trên kết quả tập phổ biến đó.

Các cột gồm số tập phổ biến, số luật thô, số luật có `Lift > 1`, số luật sau tinh gọn, support nhỏ nhất của vế kết luận và thống kê Lift.

In [ ]:
jl.seval(r'''
sweep_results = sweep_thresholds(transactions)
open("/content/sweep_results.tsv", "w") do io
    println(io, join(["minsup","minsup_count","minconf","n_itemsets","n_rules",
                      "n_lift_gt1","n_curated","min_consequent_count",
                      "median_lift","max_lift"], '\t'))
    for r in sweep_results
        println(io, join([r.minsup, r.minsup_count, r.minconf, r.n_itemsets,
                          r.n_rules, r.n_lift_gt1, r.n_curated,
                          r.min_consequent_count, r.median_lift, r.max_lift], '\t'))
    end
end
''')
sweep_df = pd.read_csv("/content/sweep_results.tsv", sep="\t")
assert len(sweep_df) == 35, f"Mong đợi 35 tổ hợp, nhận {len(sweep_df)}"
display(sweep_df.style.format({
    "minsup": "{:.3f}", "minconf": "{:.1f}",
    "median_lift": "{:.3f}", "max_lift": "{:.3f}"
}))

## 5. Khai phá luật với ngưỡng được chọn

Theo báo cáo thực nghiệm, cấu hình được chọn là `minsup = 0.005` (support count tối thiểu 50 giao dịch) và `minconf = 0.3`.

Notebook gọi `run_market_basket` để chạy Apriori và sinh luật kết hợp. Đây là pipeline có sẵn trong App 1, không có bản cài đặt thuật toán riêng trong Notebook.

In [ ]:
jl.seval("""
selected_minsup = 0.005
selected_minconf = 0.3
itemsets, rules = run_market_basket(
    transactions; minsup=selected_minsup, minconf=selected_minconf
)
""")
n_itemsets = int(jl.seval("length(itemsets)"))
n_rules = int(jl.seval("length(rules)"))
print(f"minsup = 0.005 | minconf = 0.3")
print(f"Số tập phổ biến: {n_itemsets:,}")
print(f"Số luật kết hợp sinh ra: {n_rules:,}")

## 6. Lọc, tinh gọn và xếp hạng luật

Thứ tự xử lý bám theo App 1:

1. Giữ các luật có `Lift > 1` bằng `curate_rules`.
2. Loại luật dư thừa theo heuristic đã cài trong ứng dụng: cùng vế phải nhưng có luật tổng quát hơn với Confidence không thấp hơn; đồng thời chỉ giữ một luật tốt nhất cho mỗi itemset.
3. Lấy Top 10 theo Lift và Top 10 theo Support bằng `top_rules`.

Support trong package được lưu dưới dạng **số giao dịch tuyệt đối**; cột Support (%) dưới đây được tính bằng `support / 9835 × 100`.

In [ ]:
jl.seval(r'''
curated_rules = curate_rules(rules)

function export_top_rules(path, selected, labels, n_tx)
    open(path, "w") do io
        println(io, join(["Rank","Rule","Support_count","Support_pct","Confidence","Lift"], '\t'))
        for (i, r) in enumerate(selected)
            rule_text = format_rule(r, labels; digits=3)
            println(io, join([i, rule_text, r.support,
                              100 * r.support / n_tx,
                              r.confidence, r.lift], '\t'))
        end
    end
end

top_lift = top_rules(curated_rules, 10; by=:lift)
top_support = top_rules(curated_rules, 10; by=:support)
export_top_rules("/content/top_lift.tsv", top_lift, labels, length(transactions))
export_top_rules("/content/top_support.tsv", top_support, labels, length(transactions))
''')

n_curated = int(jl.seval("length(curated_rules)"))
print(f"Số luật Lift > 1 và sau tinh gọn: {n_curated:,}")
top_lift_df = pd.read_csv("/content/top_lift.tsv", sep="\t")
top_support_df = pd.read_csv("/content/top_support.tsv", sep="\t")

display(Markdown("### Top 10 luật theo Lift"))
display(top_lift_df.style.format({
    "Support_pct": "{:.2f}%", "Confidence": "{:.3f}", "Lift": "{:.3f}"
}))
display(Markdown("### Top 10 luật theo Support"))
display(top_support_df.style.format({
    "Support_pct": "{:.2f}%", "Confidence": "{:.3f}", "Lift": "{:.3f}"
}))

## 7. Phân tích kết quả và ý nghĩa kinh doanh

### 7.1. Các luật có Lift cao

Theo kết quả thực nghiệm trong báo cáo, nhóm luật nổi bật gồm:

- `{pip fruit, root vegetables} ⇒ {other vegetables, whole milk}`: Lift khoảng **4,716**, Support 54 giao dịch (0,55%). Đây là liên hệ nổi bật nhưng xuất hiện trong tỷ lệ nhỏ giao dịch.
- `{herbs} ⇒ {root vegetables}`: Confidence khoảng **0,431**, Lift khoảng **3,956**. Có thể thử nghiệm gợi ý mua kèm nhóm rau củ khi khách chọn herbs.
- `{curd, tropical fruit} ⇒ {yogurt}`: Confidence khoảng **0,515**, Lift khoảng **3,691**. Có thể thử nghiệm gợi ý yogurt hoặc coupon đi kèm.

### 7.2. Phân biệt Lift và Support

Luật `{other vegetables} ⇒ {whole milk}` đứng đầu theo Support trong báo cáo: 736 giao dịch (7,48%), nhưng Lift khoảng 1,514. Ngược lại, luật có Lift cao nhất chỉ xuất hiện trong 54 giao dịch (0,55%).

- **Support** phù hợp để nhận diện hành vi mua phổ biến, hỗ trợ dự trữ và bố trí hàng.
- **Lift** giúp tìm các tổ hợp có mức liên kết nổi bật so với xác suất nền, phù hợp để đề xuất mua kèm hoặc thử nghiệm combo.
- Nên xem đồng thời cả hai chỉ số; Lift cao không tự động đồng nghĩa với quy mô kinh doanh lớn.

### 7.3. Lưu ý khi diễn giải

Các luật chỉ mô tả sự đồng xuất hiện trong dữ liệu quan sát. Chúng không chứng minh mua mặt hàng ở vế trái gây ra việc mua mặt hàng ở vế phải. Những đề xuất như bố trí gần nhau, combo hoặc coupon cần được kiểm chứng bằng thử nghiệm kinh doanh có đối chứng. Kết quả cũng phụ thuộc vào ngưỡng và heuristic tinh gọn được chọn.

## 8. Kiểm tra kết quả và tái hiện

Cell dưới đây đối chiếu các con số chính với kết quả đã trình bày trong báo cáo App 1. Nếu mã nguồn hoặc dữ liệu thay đổi, Notebook sẽ báo rõ thay vì âm thầm trình bày kết quả cũ.

In [ ]:
expected = {
    "transactions": 9835,
    "items": 169,
    "itemsets": 1001,
    "rules": 487,
    "curated": 348,
}
observed = {
    "transactions": n_transactions,
    "items": n_items,
    "itemsets": n_itemsets,
    "rules": n_rules,
    "curated": n_curated,
}
comparison = pd.DataFrame([
    {"Chỉ số": k, "Kỳ vọng theo báo cáo": v, "Kết quả chạy Notebook": observed[k],
     "Khớp": v == observed[k]}
    for k, v in expected.items()
])
display(comparison)
if not comparison["Khớp"].all():
    print("LƯU Ý: Có chỉ số không khớp báo cáo. Hãy kiểm tra phiên bản mã nguồn, dữ liệu và môi trường.")
else:
    print("Tất cả chỉ số kiểm tra chính khớp với báo cáo App 1.")